# Sesion 8 — De Datos Crudos a Dataset Analizable
## Diplomado: Machine Learning en Seguros · FC UNAM
### 2 de mayo de 2026  ·  07:00 - 11:00 h  (4 horas)

---

> **Premisa de la sesion:** recibes los archivos crudos de la aseguradora.
> Tu trabajo: convertirlos en un dataset limpio, bien tipado y eficiente,
> resolviendo 7 problemas reales uno por uno.

---

**Prerequisito:** Debe existir la carpeta `datos/` con los archivos CSV.

## Las 7 Dudas que Resolvemos Hoy

| # | Duda | Herramienta |
|---|------|-------------|
| 1 | 46 columnas — ¿cuales necesito? | Taxonomia + `usecols` |
| 2 | Texto sucio: M/F/Masculino/Femenino | `str` operations |
| 3 | Fechas como texto: '15/04/2026' | `pd.to_datetime()` |
| 4 | API de reaseguro devuelve JSON | `read_json()` + `json_normalize()` |
| 5 | 90k filas, 11MB sin esperar | `chunks` + `category` |
| 6 | Downcast rompio precision de primas | Estrategia segura de `dtypes` |
| 7 | ¿Cuando cambiar a Polars? | `polars` — intro y comparativa |

---
## ACT 1 — El Dataset Crudo

### Duda 1: 46 columnas — ¿cuales necesito?

In [146]:
import pandas as pd
import numpy as np
import os, time

# ── Paso 1: Cargar TODO para entender que hay ────────────────────────────────
# Primera regla: antes de descartar, entende lo que tienes
df_todo = pd.read_csv('datos/cartera_polizas.csv', nrows=5)  # solo 5 filas para ver
print(f'Columnas totales: {len(df_todo.columns)}')
print()
print('Lista de columnas:')
for i, col in enumerate(df_todo.columns, 1):
    print(f'  {i:>2}. {col}')

Columnas totales: 46

Lista de columnas:
   1. id_poliza
   2. num_poliza
   3. id_contrato_interno
   4. folio_emision
   5. id_sistema_legacy
   6. nombre
   7. apellido_paterno
   8. apellido_materno
   9. nombre_completo
  10. rfc
  11. fecha_nacimiento
  12. edad
  13. sexo
  14. estado_civil
  15. ocupacion
  16. nivel_educacion
  17. ramo
  18. plan
  19. fecha_emision
  20. fecha_inicio_vigencia
  21. fecha_fin_vigencia
  22. num_renovaciones
  23. status_poliza
  24. motivo_baja
  25. canal_venta
  26. marca_vehiculo
  27. modelo_vehiculo
  28. tipo_vehiculo
  29. suma_asegurada
  30. deducible
  31. prima_neta
  32. prima_total
  33. cuota_prima
  34. forma_pago
  35. num_cuotas
  36. agente_id
  37. estado
  38. municipio
  39. codigo_postal
  40. coord_lat
  41. coord_lon
  42. version_documento
  43. hash_documento
  44. timestamp_carga
  45. usuario_captura
  46. ip_carga


In [147]:
# ── Paso 2: Diagnostico de todas las columnas ────────────────────────────────
# Cargamos todo para el diagnostico inicial
df_full = pd.read_csv('datos/cartera_polizas.csv')
mb_full = df_full.memory_usage(deep=True).sum() / 1024**2
print(f'Dataset completo: {df_full.shape} · {mb_full:.1f} MB')
print()

# Perfil de cada columna: tipo, NaN%, valores unicos
print(f'{"Columna":<30} {"Dtype":<12} {"NaN%":>7} {"Unicos":>8}')
print('-' * 65)
for col in df_full.columns:
    dtype = str(df_full[col].dtype)
    nan_pct = df_full[col].isna().mean() * 100
    n_uniq  = df_full[col].nunique()
    print(f'{col:<30} {dtype:<12} {nan_pct:>6.1f}% {n_uniq:>8,}')

Dataset completo: (50000, 46) · 112.0 MB

Columna                        Dtype           NaN%   Unicos
-----------------------------------------------------------------
id_poliza                      object          0.0%   50,000
num_poliza                     object          0.0%   50,000
id_contrato_interno            object          0.0%   48,572
folio_emision                  object          0.0%   49,870
id_sistema_legacy              object          0.0%   49,988
nombre                         object          0.0%       55
apellido_paterno               object          0.0%       38
apellido_materno               object          0.0%       23
nombre_completo                object          0.0%   31,140
rfc                            object          0.0%   50,000
fecha_nacimiento               object          0.0%   15,676
edad                           int64           0.0%       46
sexo                           object          0.0%        4
estado_civil                   object 

In [148]:
# ── Paso 3: Clasificar las columnas por categoria ────────────────────────────
# Esta clasificacion es una DECISION DE NEGOCIO — no solo tecnica

ANALITICAS = [
    'id_poliza','num_poliza','ramo','plan','status_poliza',
    'nombre','apellido_paterno','apellido_materno',
    'rfc','edad','sexo','estado_civil','ocupacion',
    'fecha_nacimiento',
    'fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia',
    'num_renovaciones','motivo_baja',
    'suma_asegurada','deducible','prima_neta','prima_total',
    'forma_pago','agente_id','canal_venta',
    'estado','municipio','codigo_postal',
    'marca_vehiculo','modelo_vehiculo','tipo_vehiculo',  # solo Autos
]

REDUNDANTES = [
    'nombre_completo',        # = nombre + ap_pat + ap_mat
    'id_contrato_interno',    # ≈ id_poliza
    'folio_emision',          # ≈ num_poliza
    'cuota_prima',            # = prima_total / num_cuotas
    'num_cuotas',             # derivado de forma_pago
]

ADMINISTRATIVAS = [
    'hash_documento',         # hash SHA del PDF — auditoria IT
    'timestamp_carga',        # mismo valor para todos
    'ip_carga',               # IP del servidor batch
    'usuario_captura',        # operacion interna
    'version_documento',      # version del formato del contrato
    'id_sistema_legacy',      # util SOLO para joins con sistema core
]

CONDICIONALES = [
    'nivel_educacion',        # si lo necesitas para el modelo
    'coord_lat','coord_lon',  # solo si haces analisis geoespacial
]

print(f'Analiticas:     {len(ANALITICAS)}')
print(f'Redundantes:    {len(REDUNDANTES)}')
print(f'Administrativas:{len(ADMINISTRATIVAS)}')
print(f'Condicionales:  {len(CONDICIONALES)}')
print(f'Total:          {len(ANALITICAS)+len(REDUNDANTES)+len(ADMINISTRATIVAS)+len(CONDICIONALES)}')

Analiticas:     32
Redundantes:    5
Administrativas:6
Condicionales:  3
Total:          46


In [149]:
# ── Paso 4: Cargar SOLO las columnas que necesitamos ─────────────────────────
t0 = time.time()
df = pd.read_csv(
    'datos/cartera_polizas.csv',
    usecols=ANALITICAS,
    na_values=['N/D','N/A','ND','--','Sin dato',''],
)
t1 = time.time()
mb_opt = df.memory_usage(deep=True).sum() / 1024**2

print('=== COMPARATIVA DE CARGA ===')
print(f'Carga completa (46 cols):  {mb_full:.1f} MB')
print(f'Carga selectiva ({len(ANALITICAS)} cols):  {mb_opt:.1f} MB')
print(f'Reduccion de memoria:      {(1-mb_opt/mb_full)*100:.0f}%')
print(f'Tiempo de carga:           {(t1-t0)*1000:.0f} ms')
print()
print(f'Dataset de trabajo: {df.shape}')
df.head(3)

=== COMPARATIVA DE CARGA ===
Carga completa (46 cols):  112.0 MB
Carga selectiva (32 cols):  73.7 MB
Reduccion de memoria:      34%
Tiempo de carga:           797 ms

Dataset de trabajo: (50000, 32)


,id_poliza,num_poliza,nombre,apellido_paterno,apellido_materno,rfc,fecha_nacimiento,edad,sexo,estado_civil,...,tipo_vehiculo,suma_asegurada,deducible,prima_neta,prima_total,forma_pago,agente_id,estado,municipio,codigo_postal
0,POL-000001,Vid-21-000001,Gabriela,Moreno,Vega,MOGV020429CG6,29/04/2002,24,F,Union libre,...,NaN,3000000,NaN,54000.0,67651.20,Mensual,AG054,Veracruz,Poza Rica,36619.0
1,POL-000002,Aut-19-000002,Valeria,Torres,Castillo,TOVC020815IA8,15/08/2002,23,Femenino,Casado,...,Compacto,150000,8000.0,5250.0,6394.50,Trimestral,AG004,Michoacan,Morelia,58889.0
2,POL-000003,GMM-22-000003,Fernanda,Ramos,Silva,RAFS941018BC1,18/10/1994,31,M,Union libre,...,NaN,800000,5000.0,17600.0,22049.28,Mensual,AG051,Baja California,Tecate,45784.0


### 📝 Ejercicio 1 — Auditoria de columnas (8 min)

Usando el perfil que generaste arriba:
- **1a.** Identifica las 3 columnas con mas NaN — ¿tienen sentido esos NaN o son errores?
- **1b.** Encuentra columnas con menos de 5 valores unicos — ¿cuales deberian ser `category`?
- **1c.** Hay una columna numerica con un rango imposible (negativo o muy alto) — ¿cual es?
  *Pista: usa `df.describe()` sobre las columnas analiticas*

In [150]:
# Tu codigo aqui:

print("\n1a. Las 5 columnas con mas NaN:")
nan_pct = (df.isna().mean() * 100).sort_values(ascending=False)
for col, pct in nan_pct.head(5).items():
    sentido = {
        'deducible':   '✓ Esperado — polizas de Vida no tienen deducible',
        'motivo_baja': '✓ Esperado — solo polizas canceladas tienen motivo',
        'marca_vehiculo': '✓ Esperado — solo Autos tiene vehiculo',
        'modelo_vehiculo':'✓ Esperado — solo Autos tiene vehiculo',
        'tipo_vehiculo':  '✓ Esperado — solo Autos tiene vehiculo',
        'ocupacion':   '? Posible error — 8% de asegurados sin ocupacion registrada',
        'prima_neta':  '✗ Error de captura — primas faltantes requieren imputacion',
    }.get(col, '? Revisar con el area de sistemas')
    print(f"  {col:<25}: {pct:>5.1f}% NaN  → {sentido}")



1a. Las 5 columnas con mas NaN:
  motivo_baja              :  95.1% NaN  → ✓ Esperado — solo polizas canceladas tienen motivo
  marca_vehiculo           :  70.5% NaN  → ✓ Esperado — solo Autos tiene vehiculo
  tipo_vehiculo            :  70.5% NaN  → ✓ Esperado — solo Autos tiene vehiculo
  modelo_vehiculo          :  70.5% NaN  → ✓ Esperado — solo Autos tiene vehiculo
  deducible                :  15.0% NaN  → ✓ Esperado — polizas de Vida no tienen deducible


In [151]:
 # 1b: columnas candidatas a category
print("\n1b. Columnas con < 5% valores unicos → candidatas a 'category':")
for col in df.select_dtypes('object').columns:
    n_uniq = df[col].nunique()
    pct    = n_uniq / len(df) * 100
    if pct < 5:
        print(f"  {col:<25}: {n_uniq:>5} valores unicos ({pct:.2f}%)")


1b. Columnas con < 5% valores unicos → candidatas a 'category':
  nombre                   :    55 valores unicos (0.11%)
  apellido_paterno         :    38 valores unicos (0.08%)
  apellido_materno         :    23 valores unicos (0.05%)
  sexo                     :     4 valores unicos (0.01%)
  estado_civil             :     5 valores unicos (0.01%)
  ocupacion                :    20 valores unicos (0.04%)
  ramo                     :     4 valores unicos (0.01%)
  plan                     :    12 valores unicos (0.02%)
  fecha_emision            :  2190 valores unicos (4.38%)
  fecha_inicio_vigencia    :  2190 valores unicos (4.38%)
  fecha_fin_vigencia       :  2188 valores unicos (4.38%)
  status_poliza            :     4 valores unicos (0.01%)
  motivo_baja              :     4 valores unicos (0.01%)
  canal_venta              :     6 valores unicos (0.01%)
  marca_vehiculo           :    10 valores unicos (0.02%)
  tipo_vehiculo            :     8 valores unicos (0.02%)
  forma

In [152]:
 # 1c: rango imposible
print("\n1c. Diagnostico de rangos en columnas numericas:")
print(df[['edad','suma_asegurada','prima_neta','prima_total','num_renovaciones']].describe().round(2))
print("\n  Nota: num_renovaciones con 0-4 es correcto.")
print("  Verificar si prima_neta tiene valores negativos (error de captura):")
negativos = df[df['prima_neta'] < 0]['prima_neta'] if 'prima_neta' in df.columns else pd.Series()
print(f"  Primas negativas: {len(negativos)} (si hay, son errores de captura)")



1c. Diagnostico de rangos en columnas numericas:
           edad  suma_asegurada  prima_neta  prima_total  num_renovaciones
count  50000.00        50000.00    49000.00     50000.00          50000.00
mean      43.32      1002000.00    24419.54     29283.17              1.24
std       12.96      1105162.53    25474.13     30581.86              1.13
min       21.00       150000.00     2400.00      2784.00              0.00
25%       32.00       300000.00     6798.00      8120.00              0.00
50%       43.00       500000.00    14000.00     17052.00              1.00
75%       54.00      1000000.00    28000.00     33825.60              2.00
max       66.00      5000000.00   145800.00    182658.24              4.00

  Nota: num_renovaciones con 0-4 es correcto.
  Verificar si prima_neta tiene valores negativos (error de captura):
  Primas negativas: 0 (si hay, son errores de captura)


---
### Duda 2: Texto Sucio — str Operations

El campo `sexo` tiene 4 representaciones distintas del mismo valor.
Si no lo normalizas, `groupby('sexo')` produce 8 grupos en lugar de 2.

In [153]:
# ── Ver el problema ──────────────────────────────────────────────────────────
print('Valores unicos en sexo ANTES de limpiar:')
print(df['sexo'].value_counts(dropna=False))
print(f'Total valores distintos: {df["sexo"].nunique()}')
print()

# Si haces groupby ahora, obtienes grupos incorrectos:
grupos_incorrectos = df.groupby('sexo')['prima_total'].mean()
print(grupos_incorrectos.to_string())
print(f'Grupos sin limpiar: {len(grupos_incorrectos)} (deberian ser 2)')

Valores unicos en sexo ANTES de limpiar:
sexo
M            16697
F            16585
Femenino      8466
Masculino     8252
Name: count, dtype: int64
Total valores distintos: 4

sexo
F            29053.277976
Femenino     29305.291745
M            29457.558936
Masculino    29369.674515
Grupos sin limpiar: 4 (deberian ser 2)


In [154]:
# ── Solucion: normalizar con str operations ──────────────────────────────────

# Paso 1: normalizar a mayusculas sin espacios
df['sexo'] = df['sexo'].str.strip().str.upper()

# Paso 2: mapear todas las variantes al estandar
MAPA_SEXO = {
    'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
    'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',
}
df['sexo'] = df['sexo'].map(MAPA_SEXO)
# Valores no reconocidos quedan como NaN automaticamente

print('DESPUES de normalizar:')
print(df['sexo'].value_counts(dropna=False))
print()
# Ahora groupby correcto
print('Prima promedio por sexo:')
print(df.groupby('sexo')['prima_total'].mean().round(2))

DESPUES de normalizar:
sexo
F    25051
M    24949
Name: count, dtype: int64

Prima promedio por sexo:
sexo
F    29138.45
M    29428.49
Name: prima_total, dtype: float64


In [155]:
# ── Mas str operations sobre los datos reales ────────────────────────────────

# Limpiar codigo_postal: eliminar 'N/D' (ya convertido a NaN por na_values)
# Verificar:
print(f'CP con NaN: {df["codigo_postal"].isna().sum()}')
# Rellenar CP desconocido con 'DESCONOCIDO' para no perder la fila
df['codigo_postal'] = df['codigo_postal'].fillna('DESCONOCIDO')



CP con NaN: 1500


In [156]:
# Extraer ramo y anio desde num_poliza ('GMM-24-000123')
df['ramo_codigo']  = df['num_poliza'].str.extract(r'^([A-Z]+)-')
df['anio_poliza']  = df['num_poliza'].str.extract(r'-([0-9]{2})-').astype(float).astype('Int16')

# Verificar
print(df[['num_poliza','ramo_codigo','anio_poliza']].head(5).to_string(index=False))

   num_poliza ramo_codigo  anio_poliza
Vid-21-000001         NaN           21
Aut-19-000002         NaN           19
GMM-22-000003         GMM           22
Vid-19-000004         NaN           19
Vid-20-000005         NaN           20


---
### Duda 3: Fechas como Texto — pd.to_datetime()

In [157]:
# ── El problema: fechas llegaron como strings ────────────────────────────────
print('Tipo ANTES de convertir:', df['fecha_nacimiento'].dtype)

print('Muestra:', df['fecha_nacimiento'].head(3).values)
print()

# ── Convertir fecha_nacimiento (formato d/m/Y) ────────────────────────────────
df['fecha_nacimiento'] = pd.to_datetime(
    df['fecha_nacimiento'],
    format='%d/%m/%Y',
    errors='coerce'  # fechas invalidas → NaT (no detiene el proceso)
)

# ── Convertir fechas ISO estandar ────────────────────────────────────────────
for col_fecha in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
    df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce')

print('Fechas convertidas:')
print(df[['fecha_nacimiento','fecha_emision','fecha_fin_vigencia']].dtypes)
print(f'NaT en fecha_nacimiento: {df["fecha_nacimiento"].isna().sum()}')

Tipo ANTES de convertir: object
Muestra: ['29/04/2002' '15/08/2002' '18/10/1994']

Fechas convertidas:
fecha_nacimiento      datetime64[ns]
fecha_emision         datetime64[ns]
fecha_fin_vigencia    datetime64[ns]
dtype: object
NaT en fecha_nacimiento: 0


In [158]:
# ── Calculos actuariales con las fechas convertidas ──────────────────────────
hoy = pd.Timestamp.today()

# Edad calculada (mas precisa que la columna 'edad' del CSV)
df['edad_calc'] = ((hoy - df['fecha_nacimiento']).dt.days / 365.25)

# Dias de vigencia de la poliza
df['dias_vigencia'] = (df['fecha_fin_vigencia'] - df['fecha_inicio_vigencia']).dt.days

# Fraccion expuesta (cuanto del periodo ya transcurrio)
dias_transcurridos = (hoy - df['fecha_inicio_vigencia']).dt.days
df['fraccion_expuesta'] = (dias_transcurridos / df['dias_vigencia']).clip(0, 1).round(4)

# Componentes de fecha para groupby temporal
df['anio_emision']     = df['fecha_emision'].dt.year
df['mes_emision']      = df['fecha_emision'].dt.month
df['trimestre_emision']= df['fecha_emision'].dt.quarter

# Verificar
print(df[['nombre','edad','edad_calc','dias_vigencia','fraccion_expuesta']].head(5).to_string(index=False))

  nombre  edad  edad_calc  dias_vigencia  fraccion_expuesta
Gabriela    24  24.054757            365                1.0
 Valeria    23  23.759069            366                1.0
Fernanda    31  31.583847            365                1.0
  Silvia    40  40.284736            366                1.0
 Antonio    29  29.823409            365                1.0


### 📝 Ejercicio 2 — Limpiar siniestros.csv (10 min)

El archivo `siniestros.csv` tiene fechas en 3 formatos distintos:
- `fecha_ocurrencia`: YYYY-MM-DD
- `fecha_apertura`: d/m/Y (mismo dia que fecha_reporte pero formato distinto — es REDUNDANTE)
- `fecha_ultimo_movimiento`: d/m/Y

**2a.** Carga siniestros.csv usando SOLO las columnas utiles (descarta las administrativas y redundantes).
**2b.** Convierte las 3 columnas de fecha a datetime con el formato correcto.
**2c.** Calcula `dias_reporte_real` = fecha_reporte - fecha_ocurrencia.
**2d.** Calcula `dias_resolucion_real` = fecha_cierre - fecha_reporte (NaT si no esta cerrado).
**2e.** Verifica: ¿cuantos siniestros llevan mas de 180 dias sin cerrar?

In [159]:
# Tu codigo aqui:
siniestros_cols_utiles = [
    'id_siniestro','id_poliza','ramo','tipo_siniestro',
    'fecha_ocurrencia','fecha_reporte','fecha_ultimo_movimiento','fecha_cierre',
    'dias_reporte','monto_reclamado','monto_pagado',
    'status_siniestro','motivo_rechazo','id_ajustador',
]
# Carga, convierte fechas y calcula los campos derivados:
df_sin = pd.read_csv(
    'datos/siniestros.csv',
    usecols=siniestros_cols_utiles,
    na_values=['N/D','N/A','ND','--','Sin dato',''],
)

print(f"\n2a. Cargado: {df_sin.shape}  (descartamos 16 columnas administrativas/redundantes)")


2a. Cargado: (18000, 14)  (descartamos 16 columnas administrativas/redundantes)


In [160]:
 # 2b: convertir fechas
df_sin['fecha_ocurrencia'] = pd.to_datetime(df_sin['fecha_ocurrencia'], errors='coerce')
df_sin['fecha_reporte']    = pd.to_datetime(df_sin['fecha_reporte'],    errors='coerce')
df_sin['fecha_cierre']     = pd.to_datetime(df_sin['fecha_cierre'],     errors='coerce')
df_sin['fecha_ultimo_movimiento'] = pd.to_datetime(
    df_sin['fecha_ultimo_movimiento'], format='%d/%m/%Y', errors='coerce')


n_nat = df_sin['fecha_ultimo_movimiento'].isna().sum()
print(f"2b. Fechas convertidas. NaT en fecha_ultimo_movimiento: {n_nat}")

2b. Fechas convertidas. NaT en fecha_ultimo_movimiento: 0


In [161]:
df_sin['dias_reporte_real'] = (
        df_sin['fecha_reporte'] - df_sin['fecha_ocurrencia']).dt.days

In [162]:
df_sin['dias_resolucion_real'] = (
        df_sin['fecha_cierre'] - df_sin['fecha_reporte']).dt.days

---
### Duda 4: JSON — Datos de API


In [163]:
# ── Simular una respuesta JSON de una API de reaseguro ───────────────────────
import json

# Este es el tipo de JSON que recibirias de una API REST
respuesta_api = {
    'timestamp': '2026-05-02T07:00:00',
    'origen': 'Sistema_Reaseguro_v3.1',
    'polizas_reaseguradas': [
        {'id_poliza':'POL-000001','ramo':'GMM',
         'reasegurador':{'nombre':'Munich Re','participacion':0.40,'prima':960.0},
         'limites':{'maximo':2_000_000,'retencion':500_000}},
        {'id_poliza':'POL-000002','ramo':'Vida',
         'reasegurador':{'nombre':'Swiss Re','participacion':0.35,'prima':1820.0},
         'limites':{'maximo':5_000_000,'retencion':1_000_000}},
        {'id_poliza':'POL-000005','ramo':'GMM',
         'reasegurador':{'nombre':'Munich Re','participacion':0.40,'prima':550.0},
         'limites':{'maximo':2_000_000,'retencion':500_000}},
    ]
}

# Guardar como JSON (simula lo que llegaria de la API)
with open('datos/respuesta_reaseguro.json','w',encoding='utf-8') as f:
    json.dump(respuesta_api, f, ensure_ascii=False, indent=2)

print('JSON guardado en datos/respuesta_reaseguro.json')
print('Primeras lineas:')
print(json.dumps(respuesta_api, indent=2, ensure_ascii=False)[:300])

JSON guardado en datos/respuesta_reaseguro.json
Primeras lineas:
{
  "timestamp": "2026-05-02T07:00:00",
  "origen": "Sistema_Reaseguro_v3.1",
  "polizas_reaseguradas": [
    {
      "id_poliza": "POL-000001",
      "ramo": "GMM",
      "reasegurador": {
        "nombre": "Munich Re",
        "participacion": 0.4,
        "prima": 960.0
      },
      "limites": 


In [164]:
# ── Problema: JSON anidado no se carga directo en DataFrame ─────────────────
# pd.read_json funciona para JSON simple pero no para estructuras anidadas

# Cargar el JSON
with open('datos/respuesta_reaseguro.json') as f:
    data = json.load(f)

# Intentar con pd.read_json — no da el resultado esperado con anidados
# pd.read_json('datos/respuesta_reaseguro.json')  # solo lee nivel 1

# ── Solucion: json_normalize aplana el JSON anidado ──────────────────────────
from pandas import json_normalize

df_reas = json_normalize(
    data['polizas_reaseguradas'],
    sep='_'    # separador para campos anidados
)

print('DataFrame aplanado:')
print(df_reas.to_string(index=False))
print()
print('Columnas generadas:', list(df_reas.columns))

DataFrame aplanado:
 id_poliza ramo reasegurador_nombre  reasegurador_participacion  reasegurador_prima  limites_maximo  limites_retencion
POL-000001  GMM           Munich Re                        0.40               960.0         2000000             500000
POL-000002 Vida            Swiss Re                        0.35              1820.0         5000000            1000000
POL-000005  GMM           Munich Re                        0.40               550.0         2000000             500000

Columnas generadas: ['id_poliza', 'ramo', 'reasegurador_nombre', 'reasegurador_participacion', 'reasegurador_prima', 'limites_maximo', 'limites_retencion']


In [165]:
# ── json_normalize con listas anidadas ───────────────────────────────────────
# Caso mas complejo: cuando hay listas dentro del JSON

respuesta_multi = {
    'polizas': [
        {'id':'P01','coberturas':[{'tipo':'Hospitalizacion','suma':500_000},{'tipo':'Cirugia','suma':300_000}]},
        {'id':'P02','coberturas':[{'tipo':'Hospitalizacion','suma':800_000}]},
    ]
}

# record_path: donde esta la lista a 'explotar'
# meta: campos del nivel padre que quieres conservar
df_cob = json_normalize(
    respuesta_multi['polizas'],
    record_path='coberturas',
    meta=['id'],
    sep='_'
)
print(df_cob)

              tipo    suma   id
0  Hospitalizacion  500000  P01
1          Cirugia  300000  P01
2  Hospitalizacion  800000  P02


In [166]:
# ── Guardar DataFrame como JSON ──────────────────────────────────────────────

# orient='records' — lista de objetos (lo mas comun para APIs)
df.head(10).to_json('datos/muestra.json',
    orient='records',
    force_ascii=False,  # preserva caracteres especiales (acentos)
    indent=2,
    date_format='iso'   # fechas en formato ISO
)

# Verificar
with open('datos/muestra.json') as f:
    preview = f.read(300)
print(preview)

[
  {
    "id_poliza":"POL-000001",
    "num_poliza":"Vid-21-000001",
    "nombre":"Gabriela",
    "apellido_paterno":"Moreno",
    "apellido_materno":"Vega",
    "rfc":"MOGV020429CG6",
    "fecha_nacimiento":"2002-04-29T00:00:00.000",
    "edad":24,
    "sexo":"F",
    "estado_civil":"Union libre",


---
### Duda 5: 90k Filas — Procesar sin Esperar (chunks)

In [167]:
# ── Ver el tamano del archivo de beneficiarios ───────────────────────────────
mb_ben = os.path.getsize('datos/beneficiarios.csv') / 1024**2
print(f'beneficiarios.csv: {mb_ben:.1f} MB')

# Contar filas sin cargar todo
total_ben = sum(len(chunk) for chunk in
    pd.read_csv('datos/beneficiarios.csv', chunksize=10_000))
print(f'Total beneficiarios: {total_ben:,}')

# ── Patron real: calcular estadisticas por parentesco ─────────────────────────
# Sin cargar los 90k en memoria a la vez
conteo_parentesco = {}

for chunk in pd.read_csv('datos/beneficiarios.csv', chunksize=10_000):
    counts = chunk['parentesco'].value_counts().to_dict()
    for k, v in counts.items():
        conteo_parentesco[k] = conteo_parentesco.get(k, 0) + v

resultado = pd.Series(conteo_parentesco).sort_values(ascending=False)
print('Beneficiarios por parentesco (procesado por chunks):')
print(resultado)

beneficiarios.csv: 11.5 MB


Total beneficiarios: 90,000
Beneficiarios por parentesco (procesado por chunks):
Conyuge    26718
Hijo       22785
Hija       17871
Madre       7298
Padre       7173
Hermano     3596
Hermana     2765
Otro        1794
dtype: int64


In [168]:
# ── Filtrar y concatenar solo lo que necesitas ───────────────────────────────
# Obtener solo beneficiarios activos de polizas de Vida

partes = []
for chunk in pd.read_csv('datos/beneficiarios.csv',
                         chunksize=10_000,
                         usecols=['id_beneficiario','id_poliza','nombre',
                                  'apellido_paterno','parentesco','porcentaje','activo']):
    filtrado = chunk[chunk['activo'] == True]
    if len(filtrado) > 0:
        partes.append(filtrado)

ben_activos = pd.concat(partes, ignore_index=True)
print(f'Beneficiarios activos: {len(ben_activos):,}')
print(f'Memoria: {ben_activos.memory_usage(deep=True).sum()/1024:.0f} KB')

Beneficiarios activos: 67,656
Memoria: 21956 KB


---
### Duda 6: Downcast — Cuándo Es Seguro

In [169]:
# ── Demostrar el problema de precision ───────────────────────────────────────
import numpy as np

# float64 vs float32 con valores grandes
prima_grande = 15_432_756.80
print(f'Original (float64): {prima_grande}')
print(f'Como float32:       {np.float32(prima_grande)}')
print(f'Diferencia:         {abs(prima_grande - float(np.float32(prima_grande))):.2f}')
print()

# Con valores tipicos de primas individuales
prima_gmm = 3_450.75
print(f'Prima GMM (float64): {prima_gmm}')
print(f'Prima GMM (float32): {np.float32(prima_gmm)}')
print(f'Diferencia:          {abs(prima_gmm - float(np.float32(prima_gmm))):.6f}')

Original (float64): 15432756.8
Como float32:       15432757.0
Diferencia:         0.20

Prima GMM (float64): 3450.75
Prima GMM (float32): 3450.75
Diferencia:          0.000000


In [170]:
# ── Estrategia segura de optimizacion ────────────────────────────────────────
print('Memoria ANTES de optimizar:')
print(f'{df.memory_usage(deep=True).sum()/1024**2:.2f} MB')
print()

df_opt = df.copy()

# SEGURO: categoricas para columnas con pocos valores unicos
cols_category = ['ramo','plan','status_poliza','sexo','canal_venta',
                 'forma_pago','estado','estado_civil','tipo_vehiculo']
for col in cols_category:
    if col in df_opt.columns:
        n_uniq = df_opt[col].nunique()
        n_tot  = len(df_opt)
        pct = n_uniq/n_tot
        print(f'  {col:<25}: {n_uniq:>5} unicos ({pct:.1%}) → category')
        df_opt[col] = df_opt[col].astype('category')

# SEGURO: enteros pequenos
df_opt['num_renovaciones'] = df_opt['num_renovaciones'].fillna(0).astype('int8')

# SEGURO: booleano
# (activa ya no esta porque la descartamos, pero aplica el principio)

# CONSERVAR en float64: primas y sumas (montos grandes o con centavos importantes)
# NO hacer: df_opt['prima_total'] = df_opt['prima_total'].astype('float32')

print()
print('Memoria DESPUES de optimizar:')
mb_antes = df.memory_usage(deep=True).sum()/1024**2
mb_desp  = df_opt.memory_usage(deep=True).sum()/1024**2
print(f'{mb_antes:.2f} MB → {mb_desp:.2f} MB ({(1-mb_desp/mb_antes)*100:.0f}% reduccion)')

Memoria ANTES de optimizar:
67.51 MB

  ramo                     :     4 unicos (0.0%) → category
  plan                     :    12 unicos (0.0%) → category
  status_poliza            :     4 unicos (0.0%) → category
  sexo                     :     2 unicos (0.0%) → category
  canal_venta              :     6 unicos (0.0%) → category
  forma_pago               :     4 unicos (0.0%) → category
  estado                   :    15 unicos (0.0%) → category
  estado_civil             :     5 unicos (0.0%) → category
  tipo_vehiculo            :     8 unicos (0.0%) → category

Memoria DESPUES de optimizar:
67.51 MB → 41.45 MB (39% reduccion)


### 📝 Ejercicio 3 — Pipeline de limpieza completo (12 min)

Encapsula todo lo que hicimos en una funcion `limpiar_cartera(ruta_csv)` que:
- Carga con `usecols=ANALITICAS`
- Normaliza `sexo` con str + map
- Convierte fechas con `to_datetime` + `errors='coerce'`
- Calcula `edad_calc`, `dias_vigencia`, `fraccion_expuesta`
- Aplica optimizacion de memoria (categoricas)
- Retorna el DataFrame limpio

Al final llama: `df_limpio = limpiar_cartera('datos/cartera_polizas.csv')`
y verifica que no tenga texto sucio en sexo.

In [171]:
def cargar_cartera(ruta='datos/cartera_polizas.csv', cols=None):
    """Carga la cartera con las columnas correctas y na_values."""
    return pd.read_csv(
        ruta,
        usecols=cols or ANALITICAS,
        na_values=['N/D','N/A','ND','--','Sin dato',''],
    )


In [172]:
def normalizar_texto(df):
    """Normaliza campos de texto sucios."""
    if 'sexo' in df.columns:
        df['sexo'] = df['sexo'].str.strip().str.upper().map(MAPA_SEXO)
    if 'codigo_postal' in df.columns:
        df['codigo_postal'] = df['codigo_postal'].replace('N/D', np.nan)
    return df


In [173]:
def limpiar_fechas(df):
    """Convierte todas las columnas de fecha a datetime."""
    # Fecha nacimiento: formato d/m/Y (sistema legacy)
    if 'fecha_nacimiento' in df.columns:
        df['fecha_nacimiento'] = pd.to_datetime(
            df['fecha_nacimiento'], format='%d/%m/%Y', errors='coerce')

    # Fechas ISO estandar
    for col in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    # Fechas en formato d/m/Y en siniestros
    for col in ['fecha_apertura','fecha_ultimo_movimiento']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='coerce')

    # Fechas ISO en siniestros
    for col in ['fecha_ocurrencia','fecha_reporte','fecha_cierre']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    return df


In [174]:
def optimizar_tipos(df, umbral_cat=0.05, verbose=True):
    """
    Optimiza tipos de dato de forma segura.
    - Categoricas: columnas texto con < umbral_cat% de valores unicos
    - Enteros: downcast seguro segun rango
    - Floats de montos: SE CONSERVAN en float64
    """
    df = df.copy()
    mb_antes = df.memory_usage(deep=True).sum() / 1024**2

    # Categoricas
    for col in df.select_dtypes('object').columns:
        pct = df[col].nunique() / len(df)
        if pct < umbral_cat:
            df[col] = df[col].astype('category')

    # Enteros: downcast seguro
    for col in df.select_dtypes('int64').columns:
        mx = df[col].abs().max()
        if mx <= 127:    df[col] = df[col].astype('int8')
        elif mx <= 32767:df[col] = df[col].astype('int16')
        elif mx <= 2_147_483_647: df[col] = df[col].astype('int32')

    # Columnas de monto: NO hacer downcast a float32
    # (pueden tener valores > 100k con centavos importantes)

    mb_desp = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        reduccion = (1 - mb_desp/mb_antes)*100
        print(f"  Memoria: {mb_antes:.1f} MB → {mb_desp:.1f} MB ({reduccion:.0f}% reduccion)")
    return df

In [175]:
# Tu codigo aqui:
def limpiar_cartera(ruta_csv):
    """
    Pipeline completo de limpieza de la cartera.

    Pasos:
    1. Carga selectiva (solo columnas analiticas)
    2. Elimina duplicados
    3. Normaliza sexo
    4. Convierte fechas
    5. Imputa prima_neta por mediana de ramo
    6. Limpia codigo_postal
    7. Optimiza tipos de dato

    Retorna: DataFrame limpio y optimizado
    """
    print("  [1/7] Cargando columnas analiticas...")
    mb_csv = os.path.getsize(ruta_csv) / 1024**2
    df = cargar_cartera(ruta_csv)
    mb_cargado = df.memory_usage(deep=True).sum() / 1024**2
    print(f"        CSV: {mb_csv:.1f} MB → en memoria: {mb_cargado:.1f} MB · Shape: {df.shape}")

    print("  [2/7] Eliminando duplicados...")
    n_antes = len(df)
    df = df.drop_duplicates()
    print(f"        Eliminados: {n_antes - len(df)} duplicados · Quedan: {len(df):,}")

    print("  [3/7] Normalizando texto (sexo, codigo_postal)...")
    df = normalizar_texto(df)
    sexo_nans = df['sexo'].isna().sum()
    print(f"        Sexo normalizado. Valores no reconocidos → NaN: {sexo_nans}")

    print("  [4/7] Convirtiendo fechas...")
    df = limpiar_fechas(df)
    nat_total = sum(df[c].isna().sum()
                    for c in df.columns
                    if pd.api.types.is_datetime64_any_dtype(df[c]))
    print(f"        NaT generados por fechas invalidas: {nat_total}")

    print("  [5/7] Imputando prima_neta por mediana de ramo...")
    n_nan_prima = df['prima_neta'].isna().sum()
    df['prima_neta'] = df.groupby('ramo')['prima_neta'].transform(
        lambda x: x.fillna(x.median()))
    print(f"        Imputadas: {n_nan_prima} primas → mediana de su ramo")

    print("  [6/7] Limpiando campos adicionales...")
    df['codigo_postal'] = df['codigo_postal'].replace('N/D', np.nan)
    cp_nan = df['codigo_postal'].isna().sum()
    print(f"        CPs invalidos convertidos a NaN: {cp_nan}")

    print("  [7/7] Optimizando tipos de dato...")
    df = optimizar_tipos(df, verbose=True)

    # Verificacion final
    total_nan = df.isna().sum().sum()
    print(f"\n  ✓ Verificacion final:")
    print(f"    Shape:           {df.shape}")
    print(f"    NaN restantes:   {total_nan} (normales: deducible Vida, vehiculo no-Autos, etc.)")
    print(f"    Duplicados:      {df.duplicated().sum()}")
    print(f"    Sexo sucio:      {df['sexo'].isin(['MASCULINO','FEMENINO','m','f']).sum()}")
    print(f"    Fechas como str: {df.select_dtypes('object').filter(like='fecha').shape[1]}")
    return df

In [176]:
print("  SOLUCION EJERCICIO 3 — Pipeline limpiar_cartera()")
print("=" * 65)
print()
df = limpiar_cartera('datos/cartera_polizas.csv')
print()
print("  Primeras 3 filas del dataset limpio:")
print(df[['id_poliza','ramo','sexo','fecha_emision','prima_neta']].head(3).to_string(index=False))

  SOLUCION EJERCICIO 3 — Pipeline limpiar_cartera()

  [1/7] Cargando columnas analiticas...
        CSV: 20.6 MB → en memoria: 73.7 MB · Shape: (50000, 32)
  [2/7] Eliminando duplicados...
        Eliminados: 0 duplicados · Quedan: 50,000
  [3/7] Normalizando texto (sexo, codigo_postal)...
        Sexo normalizado. Valores no reconocidos → NaN: 0
  [4/7] Convirtiendo fechas...
        NaT generados por fechas invalidas: 0
  [5/7] Imputando prima_neta por mediana de ramo...
        Imputadas: 1000 primas → mediana de su ramo
  [6/7] Limpiando campos adicionales...
        CPs invalidos convertidos a NaN: 1500
  [7/7] Optimizando tipos de dato...
  Memoria: 62.3 MB → 14.4 MB (77% reduccion)

  ✓ Verificacion final:
    Shape:           (50000, 32)
    NaN restantes:   166289 (normales: deducible Vida, vehiculo no-Autos, etc.)
    Duplicados:      0
    Sexo sucio:      0
    Fechas como str: 0

  Primeras 3 filas del dataset limpio:
 id_poliza  ramo sexo fecha_emision  prima_neta
POL-00

---
### Duda 7: ¿Cuándo Cambiar a Polars?

In [177]:
# ── Instalar polars si no esta disponible ────────────────────────────────────
# En tu terminal: pip install polars
# Verificar:
try:
    import polars as pl
    print(f'Polars {pl.__version__} disponible')
    POLARS_OK = True
except ImportError:
    print('Polars no instalado. Ejecuta: pip install polars')
    POLARS_OK = False

Polars no instalado. Ejecuta: pip install polars


In [178]:
# ── Comparativa de velocidad pandas vs polars ────────────────────────────────
if POLARS_OK:
    import polars as pl
    import time

    # ── Pandas ──────────────────────────────────────────────────────────────
    t0 = time.time()
    df_pd = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS)
    res_pd = (df_pd.groupby('ramo')
                   .agg(polizas=('id_poliza','count'),
                        prima_total=('prima_total','sum'),
                        prima_prom=('prima_neta','mean'))
                   .round(2).reset_index())
    t_pd = time.time()-t0

    # ── Polars ──────────────────────────────────────────────────────────────
    t0 = time.time()
    df_pl = pl.read_csv('datos/cartera_polizas.csv', columns=ANALITICAS)
    res_pl = (df_pl
        .group_by('ramo')
        .agg([
            pl.col('id_poliza').count().alias('polizas'),
            pl.col('prima_total').sum().alias('prima_total'),
            pl.col('prima_neta').mean().alias('prima_prom'),
        ])
    )
    t_pl = time.time()-t0

    print(f'Pandas:  {t_pd*1000:.0f} ms')
    print(f'Polars:  {t_pl*1000:.0f} ms')
    print(f'Polars es {t_pd/t_pl:.1f}x mas rapido en esta operacion')
    print()
    print('Resultado Polars:')
    print(res_pl)
else:
    print('Instala polars para ver la comparativa: pip install polars')

Instala polars para ver la comparativa: pip install polars


In [179]:
# ── Sintaxis Polars: las operaciones mas comunes ──────────────────────────────
if POLARS_OK:
    df_pl = pl.read_csv('datos/cartera_polizas.csv',
                        columns=['id_poliza','ramo','prima_total','edad'])

    # Filtrar
    print('Polizas con prima > 10,000:')
    print(df_pl.filter(pl.col('prima_total') > 10_000).shape)

    # Agregar columna
    df_pl2 = df_pl.with_columns(
        (pl.col('prima_total')/12).alias('prima_mensual')
    )

    # Sort
    print('Top 5 primas:')
    print(df_pl2.sort('prima_total', descending=True).head(5)[['id_poliza','ramo','prima_total']])

    # Lazy evaluation — ejecuta todo optimizado al final
    resultado = (
        pl.scan_csv('datos/cartera_polizas.csv')  # NO carga en memoria aun
        .filter(pl.col('prima_total') > 5000)
        .group_by('ramo')
        .agg(pl.col('prima_total').sum())
        .collect()  # AHORA ejecuta con el plan optimizado
    )
    print('Lazy evaluation result:')
    print(resultado)

### Regla practica: ¿Pandas o Polars?

| Situacion | Usa |
|-----------|-----|
| Aprendizaje, primeros modelos | **pandas** — el ecosistema es enorme |
| < 1 millon de filas | **pandas** — mas que suficiente |
| Analisis interactivo en notebook | **pandas** — syntax mas conocida |
| Pipeline de produccion > 5M filas | **polars** — 5-20x mas rapido |
| Ingesta diaria de datos grandes | **polars** con lazy evaluation |
| ETL de empresa | **polars** o **spark** segun el tamano |

---
## pivot_table con Datos Reales


In [180]:
# ── Agregar columnas necesarias para el pivot ────────────────────────────────
df_work = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS,
                      na_values=['N/D','N/A',''])
df_work['prima_neta'] = df_work['prima_neta'].fillna(df_work.groupby('ramo')['prima_neta'].transform('median'))
df_work['g_edad'] = pd.cut(df_work['edad'], bins=[0,30,45,60,100],
                           labels=['18-30','31-45','46-60','61+'])
df_work['siniest_flag'] = (df_work['prima_neta'] > df_work['prima_neta'].quantile(0.75)).astype(int)

print(f'Dataset para pivot: {df_work.shape}')

Dataset para pivot: (50000, 34)


In [181]:
# ── pivot_table: prima por ramo x grupo de edad ──────────────────────────────
tabla_prima = pd.pivot_table(
    df_work,
    values   = 'prima_total',
    index    = 'ramo',
    columns  = 'g_edad',
    aggfunc  = 'sum',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
).round(0) / 1_000  # en miles de pesos

print('Prima total por ramo y grupo de edad (miles MXN):')
print(tabla_prima.to_string())

Prima total por ramo y grupo de edad (miles MXN):
g_edad                      18-30       31-45       46-60         61+        TOTAL
ramo                                                                              
Accidentes Personales    5248.369    8348.395    8731.668    2652.247    24980.679
Autos                   55457.022   84225.512   84571.404   29569.569   253823.506
GMM                    139278.258  223908.688  264218.298  103617.078   731022.322
Vida                    79667.849  134774.641  166301.773   73587.860   454332.123
TOTAL                  279651.498  451257.236  523823.143  209426.754  1464158.631


/var/folders/0x/kxtns7515ps4lbrgywb868nw0000gn/T/ipykernel_90250/3938379720.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  tabla_prima = pd.pivot_table(


In [182]:
# ── pivot_table: polizas por zona x canal ────────────────────────────────────
tabla_canal = pd.pivot_table(
    df_work,
    values   = 'id_poliza',
    index    = 'estado',
    columns  = 'canal_venta',
    aggfunc  = 'count',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
)

print('Polizas por estado y canal de venta:')
print(tabla_canal.to_string())

Polizas por estado y canal de venta:
canal_venta       Agente  Banca Seguros  Broker  Digital  Directo  Promotor  TOTAL
estado                                                                            
Baja California     1646            329     678      241      356        98   3348
CDMX                1719            311     684      217      318       108   3357
Chihuahua           1656            344     642      250      357       101   3350
Coahuila            1721            349     672      199      329        96   3366
Estado de Mexico    1703            336     652      253      287        93   3324
Guanajuato          1745            339     669      237      359        96   3445
Jalisco             1664            322     670      258      332       106   3352
Michoacan           1621            327     711      217      319        99   3294
Nuevo Leon          1681            335     615      225      326        96   3278
Puebla              1586            330     672   

---
## Exportar — CSV, Excel, Parquet y JSON


In [183]:
# ── Guardar en todos los formatos y comparar ──────────────────────────────────
df_export = df_work.head(10_000)  # subconjunto para demo

formatos = {
    'CSV':     ('datos/export_demo.csv',
                lambda: df_export.to_csv('datos/export_demo.csv', index=False)),
    'Parquet': ('datos/export_demo.parquet',
                lambda: df_export.to_parquet('datos/export_demo.parquet', index=False)),
    'JSON':    ('datos/export_demo.json',
                lambda: df_export.to_json('datos/export_demo.json',
                                          orient='records', force_ascii=False)),
}

print(f'{"Formato":<10} {"Tamano":>10} {"Tiempo":>10}')
print('-' * 35)
for nombre, (ruta, guardar) in formatos.items():
    t0 = time.time()
    guardar()
    t = (time.time()-t0)*1000
    kb = os.path.getsize(ruta)/1024
    print(f'{nombre:<10} {kb:>8.0f} KB {t:>8.0f} ms')

Formato        Tamano     Tiempo
-----------------------------------


CSV            2455 KB      289 ms
Parquet         633 KB      329 ms
JSON           7645 KB      134 ms


In [184]:
# ── Excel multihoja — el entregable mas solicitado en empresas ───────────────
resumen_ramo = df_work.groupby('ramo').agg(
    polizas    =('id_poliza','count'),
    prima_total=('prima_total','sum'),
    prima_prom =('prima_total','mean'),
).round(2).reset_index()
resumen_ramo['pct_cartera'] = (resumen_ramo['prima_total']/resumen_ramo['prima_total'].sum()*100).round(1)

with pd.ExcelWriter('datos/reporte_demo.xlsx', engine='openpyxl') as writer:
    df_work.head(5_000).to_excel(writer, sheet_name='Cartera', index=False)
    resumen_ramo.to_excel(writer, sheet_name='Resumen_Ramo', index=False)
    tabla_prima.to_excel(writer, sheet_name='Pivot_Prima')
    tabla_canal.to_excel(writer, sheet_name='Pivot_Canal')

kb_xl = os.path.getsize('datos/reporte_demo.xlsx')/1024
print(f'Excel multihoja generado: {kb_xl:.0f} KB con 4 hojas')

Excel multihoja generado: 972 KB con 4 hojas


---
## 🏆 Ejercicio Integrador Final — Pipeline Completo

**Contexto:** Tu jefa de estadistica te pide el reporte ejecutivo del Q1 2026.
Tienes los 4 archivos crudos. Debes construir un pipeline completo,
documentando cada decision de limpieza.

**Tiempo:** 40 minutos  |  **Entregable:** Excel con 5 hojas + Parquet

---

### Criterios de evaluacion:
- ✅ Cada decision de descarte de columnas esta justificada con comentario
- ✅ No hay texto sucio en columnas categoricas clave
- ✅ Todas las fechas son datetime, no object
- ✅ dtypes optimizados sin perder precision en montos
- ✅ El Excel tiene las 5 hojas con los datos correctos
- ✅ El Parquet es mas pequeno que el CSV equivalente

In [185]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 1: INGESTA INTELIGENTE
# ══════════════════════════════════════════════════════════════════════════════
# Carga cartera, siniestros y catalogo de ramos/agentes.
# Usa SOLO las columnas que necesitas — justifica con comentario.
# Mide la memoria ahorrada vs cargar todo.

print("\n[FASE 1] Ingesta inteligente")

ANALITICAS = [
    'id_poliza',
    'num_poliza',
    'nombre',
    'apellido_paterno',
    'apellido_materno',
    'rfc',
    'fecha_nacimiento',
    'edad',
    'sexo',
    'estado_civil',
    'ocupacion',
    'ramo',
    'plan',
    'fecha_emision',
    'fecha_inicio_vigencia',
    'fecha_fin_vigencia',
    'num_renovaciones',
    'status_poliza',
    'canal_venta',
    'suma_asegurada',
    'prima_neta',
    'prima_total',
    'forma_pago',
    'agente_id',
    'estado',
    'municipio',
    'codigo_postal'
]

df = pd.read_csv(
    'datos/cartera_polizas.csv',
    usecols=ANALITICAS,
    parse_dates=[
        'fecha_nacimiento',
        'fecha_emision',
        'fecha_inicio_vigencia',
        'fecha_fin_vigencia'
    ]
)

ramos = pd.read_csv('datos/catalogo_ramos.csv')

agentes = pd.read_csv('datos/catalogo_agentes.csv')

mb_selectivo = df.memory_usage(deep=True).sum() / 1024**2

df_full = pd.read_csv('datos/cartera_polizas.csv')

mb_full = df_full.memory_usage(deep=True).sum() / 1024**2

del df_full

print(f"  Carga completa (46 cols):  {mb_full:.1f} MB")
print(f"  Carga selectiva ({len(ANALITICAS)} cols): {mb_selectivo:.1f} MB")



[FASE 1] Ingesta inteligente


/var/folders/0x/kxtns7515ps4lbrgywb868nw0000gn/T/ipykernel_90250/1285019338.py:40: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df = pd.read_csv(


  Carga completa (46 cols):  112.0 MB
  Carga selectiva (27 cols): 58.7 MB


In [186]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 2: LIMPIEZA COMPLETA
# ══════════════════════════════════════════════════════════════════════════════
# 2a. Elimina duplicados de cartera
# 2b. Normaliza sexo con str.strip().str.upper() + .map(MAPA_SEXO)
# 2c. Convierte TODAS las fechas a datetime con errors='coerce'
# 2d. Rellena NaN de prima_neta con la MEDIANA POR RAMO (no global)
#     df.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
# 2e. Limpia codigo_postal (reemplaza 'N/D' con NaN)
# 2f. Aplica optimizacion de categoricas (sin tocar float64 de primas)

print("\n[FASE 2] Limpieza completa")

# Eliminar duplicados
antes = len(df)

df = df.drop_duplicates()

despues = len(df)

print(f"  Duplicados eliminados: {antes - despues}")

# Limpieza de texto
cols_texto = [
    'sexo',
    'estado_civil',
    'ocupacion',
    'ramo',
    'canal_venta',
    'estado'
]

for col in cols_texto:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
    )

# Imputación de prima_neta con mediana por ramo
df['prima_neta'] = (
    df.groupby('ramo')['prima_neta']
    .transform(lambda x: x.fillna(x.median()))
)

# Optimización de tipos
float_cols = [
    'prima_neta',
    'prima_total',
    'suma_asegurada'
]

for col in float_cols:
    df[col] = df[col].astype('float32')

int_cols = [
    'edad',
    'num_renovaciones'
]

for col in int_cols:
    df[col] = df[col].astype('int16')

mem_final = df.memory_usage(deep=True).sum() / 1024**2

print(f"  Memoria final: {mem_final:.1f} MB")



[FASE 2] Limpieza completa
  Duplicados eliminados: 0
  Memoria final: 57.7 MB


In [187]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 3: ENRIQUECIMIENTO
# ══════════════════════════════════════════════════════════════════════════════
# 3a. Merge con catalogo_ramos: agregar nombre_largo, tasa_base
# 3b. Merge con catalogo_agentes: agregar nombre del agente, region
# 3c. Crear: g_edad con pd.cut
# 3d. Crear: prima_calc = suma_asegurada * tasa_base * 1.16
# 3e. Crear: nivel_riesgo con .apply(clasificar_riesgo) — de mi_modulo
# 3f. Crear: edad_calc desde fecha_nacimiento
# 3g. Crear: dias_vigencia, fraccion_expuesta

from mi_modulo import clasificar_riesgo

def enriquecer_cartera(df, ramos, agentes):

    # Merge ramos
    df = df.merge(
        ramos[['ramo', 'nombre_largo', 'tasa_base']],
        on='ramo',
        how='left'
    )

    # Merge agentes
    agentes = agentes.rename(columns={'nombre': 'nombre_agente'})

    df = df.merge(
        agentes[['agente_id', 'nombre_agente', 'region']],
        on='agente_id',
        how='left'
    )

    # Grupo edad
    df['g_edad'] = pd.cut(
        df['edad'],
        bins=[0, 30, 45, 60, 100],
        labels=['18-30', '31-45', '46-60', '61+']
    )

    # Prima calculada
    df['prima_calc'] = (
        df['suma_asegurada'] *
        df['tasa_base'] *
        1.16
    )

    # Nivel riesgo
    df['nivel_riesgo'] = (
        df['num_renovaciones']
        .apply(clasificar_riesgo)
    )

    # Edad calculada
    hoy = pd.Timestamp.today()

    df['edad_calc'] = (
        (hoy - pd.to_datetime(df['fecha_nacimiento']))
        .dt.days // 365
    )

    # Días vigencia
    df['dias_vigencia'] = (
        pd.to_datetime(df['fecha_fin_vigencia']) -
        pd.to_datetime(df['fecha_inicio_vigencia'])
    ).dt.days

    # Fracción expuesta
    df['fraccion_expuesta'] = (
        df['dias_vigencia'] / 365
    )

    return df


df = enriquecer_cartera(df, ramos, agentes)

In [188]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 4: ANALISIS Y REPORTES
# ══════════════════════════════════════════════════════════════════════════════
# 4a. groupby+agg por ramo: polizas, prima_total, prima_prom, pct_cartera
# 4b. groupby+agg por agente: polizas, prima_total, comision (10%)
# 4c. pivot_table prima por ramo x g_edad con margins=True
# 4d. pivot_table polizas por estado x ramo
# 4e. Identifica: ramo con mayor prima total y zona con mayor frecuencia

print("\n[FASE 4] Analisis multidimensional")

# Resumen por ramo
resumen_ramo = (
    df.groupby('nombre_largo')
    .agg(
        polizas=('id_poliza', 'count'),
        prima_total=('prima_total', 'sum'),
        prima_prom=('prima_total', 'mean'),
        prima_calc_tot=('prima_calc', 'sum')
    )
    .round(2)
    .reset_index()
)

resumen_ramo['pct_cartera'] = (
    resumen_ramo['prima_total'] /
    resumen_ramo['prima_total'].sum() * 100
).round(1)

# Resumen por agente
resumen_agente = (
    df.groupby('nombre_agente')
    .agg(
        polizas=('id_poliza', 'count'),
        prima_total=('prima_total', 'sum')
    )
    .round(2)
    .reset_index()
)

resumen_agente['comision_est'] = (
    resumen_agente['prima_total'] * 0.10
).round(2)

# Pivot prima
pivot_prima = pd.pivot_table(
    df,
    values='prima_total',
    index='nombre_largo',
    columns='g_edad',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='TOTAL'
)

# Pivot zona
pivot_zona = pd.pivot_table(
    df,
    values='id_poliza',
    index='estado',
    columns='ramo',
    aggfunc='count',
    fill_value=0,
    margins=True,
    margins_name='TOTAL'
)

print("  Reportes generados")



[FASE 4] Analisis multidimensional
  Reportes generados


/var/folders/0x/kxtns7515ps4lbrgywb868nw0000gn/T/ipykernel_90250/3747641701.py:46: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_prima = pd.pivot_table(


In [189]:
# ══════════════════════════════════════════════════════════════════════════════
# INTEGRACION DE SINIESTROS
# ══════════════════════════════════════════════════════════════════════════════

print("\n[EXTRA] Integracion de siniestros")

sins = pd.read_csv(
    'datos/siniestros.csv',
    usecols=[
        'id_poliza',
        'id_siniestro',
        'monto_reclamado',
        'monto_pagado',
        'status_siniestro'
    ]
)

# Resumen por póliza
resumen_sin = sins.groupby('id_poliza').agg(
    n_siniestros   = ('id_siniestro', 'count'),
    monto_reclamado= ('monto_reclamado', 'sum'),
    monto_pagado   = ('monto_pagado', 'sum')
).reset_index()

# Merge con cartera
df = df.merge(
    resumen_sin,
    on='id_poliza',
    how='left'
)

# Llenar NaN
cols_fill = [
    'n_siniestros',
    'monto_reclamado',
    'monto_pagado'
]

df[cols_fill] = df[cols_fill].fillna(0)

# Loss ratio
df['loss_ratio'] = (
    df['monto_pagado'] /
    df['prima_total']
)

print("  Siniestros integrados")

print(df[
    [
        'id_poliza',
        'n_siniestros',
        'monto_pagado',
        'loss_ratio'
    ]
].head())


[EXTRA] Integracion de siniestros
  Siniestros integrados
    id_poliza  n_siniestros  monto_pagado  loss_ratio
0  POL-000001           0.0           0.0         0.0
1  POL-000002           0.0           0.0         0.0
2  POL-000003           0.0           0.0         0.0
3  POL-000004           0.0           0.0         0.0
4  POL-000005           1.0           0.0         0.0


In [190]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 5: EXPORTAR
# ══════════════════════════════════════════════════════════════════════════════
# Excel con 5 hojas: Cartera_Limpia, Resumen_Ramo, Resumen_Agente,
#                    Pivot_Prima, Pivot_Zona
# Parquet: cartera_q1_2026_final.parquet
# Compara tamano CSV equivalente vs Parquet

print("\n[FASE 5] Exportar resultados")

# Excel
ruta_xl = 'datos/reporte_ejecutivo_Q1_2026.xlsx'

with pd.ExcelWriter(ruta_xl, engine='openpyxl') as writer:

    cols_excel = [
        'id_poliza',
        'ramo',
        'nombre_largo',
        'g_edad',
        'nivel_riesgo',
        'prima_neta',
        'prima_total',
        'prima_calc',
        'fraccion_expuesta',
        'estado',
        'municipio',
        'nombre_agente',
        'canal_venta'
    ]

    cols_excel = [c for c in cols_excel if c in df.columns]

    df[cols_excel].to_excel(
        writer,
        sheet_name='Cartera_Limpia',
        index=False
    )

    resumen_ramo.to_excel(
        writer,
        sheet_name='Resumen_Ramo',
        index=False
    )

    resumen_agente.to_excel(
        writer,
        sheet_name='Resumen_Agente',
        index=False
    )

    pivot_prima.to_excel(
        writer,
        sheet_name='Pivot_Prima'
    )

    pivot_zona.to_excel(
        writer,
        sheet_name='Pivot_Zona'
    )

kb_xl = os.path.getsize(ruta_xl) / 1024

print(f"  Excel generado: {kb_xl:.0f} KB")


# Parquet
ruta_pq = 'datos/cartera_q1_2026_final.parquet'

df.to_parquet(
    ruta_pq,
    index=False
)

kb_pq = os.path.getsize(ruta_pq) / 1024

# CSV comparación
ruta_csv = 'datos/cartera_q1_2026_final.csv'

df.to_csv(
    ruta_csv,
    index=False
)

kb_csv = os.path.getsize(ruta_csv) / 1024

print(f"  Parquet: {kb_pq:.0f} KB")

print(f"  CSV: {kb_csv:.0f} KB")

print(f"  Parquet es {(kb_csv/kb_pq):.1f}x mas compacto")

# Limpiar CSV temporal
os.remove(ruta_csv)

# Tiempo total
t_total = time.time() - t0

print(f"\nPipeline completo en {t_total:.1f} segundos")

print(f"Dataset final: {df.shape[0]:,} filas · {df.shape[1]} columnas")



[FASE 5] Exportar resultados
  Excel generado: 3356 KB
  Parquet: 3296 KB
  CSV: 16317 KB
  Parquet es 5.0x mas compacto

Pipeline completo en 31.0 segundos
Dataset final: 50,000 filas · 41 columnas


In [191]:
print(df.columns)

Index(['id_poliza', 'num_poliza', 'nombre', 'apellido_paterno',
       'apellido_materno', 'rfc', 'fecha_nacimiento', 'edad', 'sexo',
       'estado_civil', 'ocupacion', 'ramo', 'plan', 'fecha_emision',
       'fecha_inicio_vigencia', 'fecha_fin_vigencia', 'num_renovaciones',
       'status_poliza', 'canal_venta', 'suma_asegurada', 'prima_neta',
       'prima_total', 'forma_pago', 'agente_id', 'estado', 'municipio',
       'codigo_postal', 'nombre_largo', 'tasa_base', 'nombre_agente', 'region',
       'g_edad', 'prima_calc', 'nivel_riesgo', 'edad_calc', 'dias_vigencia',
       'fraccion_expuesta', 'n_siniestros', 'monto_reclamado', 'monto_pagado',
       'loss_ratio'],
      dtype='object')


In [192]:
# ── Solucion resumida (descomenta solo si necesitas referencia) ──────────────

# FASE 1:
# df = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS, na_values=['N/D','N/A'])
# ramos_cat = pd.read_csv('datos/catalogo_ramos.csv')
# agentes_cat = pd.read_csv('datos/catalogo_agentes.csv')

# FASE 2:
# df = df.drop_duplicates()
# df['sexo'] = df['sexo'].str.strip().str.upper().map(MAPA_SEXO)
# for col in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
#     df[col] = pd.to_datetime(df[col], errors='coerce')
# df['fecha_nacimiento'] = pd.to_datetime(df['fecha_nacimiento'],format='%d/%m/%Y',errors='coerce')
# df['prima_neta'] = df.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
# df['codigo_postal'] = df['codigo_postal'].replace('N/D', np.nan)
# for col in ['ramo','plan','canal_venta','forma_pago','estado','sexo','tipo_vehiculo']:
#     if col in df.columns: df[col] = df[col].astype('category')

# FASE 3 (extracto):
# df = pd.merge(df, ramos_cat[['ramo','nombre_largo','tasa_base']], on='ramo', how='left')
# df['g_edad'] = pd.cut(df['edad'],bins=[0,30,45,60,100],labels=['18-30','31-45','46-60','61+'])
# df['prima_calc'] = df['suma_asegurada'] * df['tasa_base'] * 1.16
# from mi_modulo import clasificar_riesgo
# df['nivel_riesgo'] = df['prima_calc'].apply(lambda p: 'ALTO' if p>15000 else 'MEDIO' if p>6000 else 'BAJO')

# FASE 5:
# with pd.ExcelWriter('datos/reporte_ejecutivo_Q1_2026.xlsx', engine='openpyxl') as w:
#     df.to_excel(w,'Cartera_Limpia',index=False)
#     resumen_ramo.to_excel(w,'Resumen_Ramo',index=False)
#     resumen_agente.to_excel(w,'Resumen_Agente',index=False)
#     tabla_prima.to_excel(w,'Pivot_Prima')
#     tabla_zona.to_excel(w,'Pivot_Zona')
# df.to_parquet('datos/cartera_q1_2026_final.parquet', index=False)
# print(f'CSV: {df.to_csv(index=False).encode().__len__()/1024:.0f} KB')
# print(f'Parquet: {os.path.getsize("datos/cartera_q1_2026_final.parquet")/1024:.0f} KB')

---
## Resumen: Lo que Aprendiste en la Sesion 8

| Duda | Herramienta | Aprendizaje clave |
|------|-------------|-------------------|
| 46 columnas | `usecols` + taxonomia | Clasificar antes de cargar — decision de negocio |
| Texto sucio | `str.strip/upper/map` | Normalizar ANTES del primer groupby |
| Fechas texto | `pd.to_datetime(errors='coerce')` | Nunca detener el pipeline por fechas invalidas |
| JSON anidado | `json_normalize(sep='_')` | Aplanar antes de analizar |
| 90k filas | `chunksize` + `category` | Medir memoria antes de decidir |
| Downcast riesgoso | Regla float32 < 100k MXN | float64 para montos grandes siempre |
| Polars | `pl.scan_csv().collect()` | Lazy evaluation = optimizacion automatica |

**T5 Pandas — COMPLETADO**

**Proxima sesion — Mie 6 mayo, 18:00 h:**
T6 Visualizacion — Matplotlib, Seaborn y Plotly.

```bash
git add sesion8_M1_notebook.ipynb
git commit -m "Sesion 8: pipeline completo datos reales - str datetime JSON Polars"
git push
```

---
*Diplomado ML en Seguros · FC UNAM · 2026*